# 02. PyTorch Dataset/DataLoader로 영상 배치 만들기

Food101에서 `torchvision.datasets.ImageFolder` + `DataLoader`로 이미지를 배치로 불러왔던 것처럼,
여기서는 `01_video_io_and_frame_sampling.ipynb`에서 받아둔 `data/UCF101_subset`을 읽는
커스텀 `Dataset`을 만든다.

영상은 이미지와 달리 **(1) 압축 해제/프레임 샘플링 비용이 크고 (2) 샘플마다 길이가 다르다**는
차이가 있어서, 보통 `ImageFolder`처럼 간단하게 쓰지 못하고 커스텀 Dataset을 직접 만든다.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from video_utils import VideoClipDataset, denormalize_for_display

DATA_DIR = Path("data/UCF101_subset")
CLASSES = sorted(p.name for p in (DATA_DIR / "train").iterdir() if p.is_dir())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
print(class_to_idx)


## 2-1. (경로, 라벨) 목록 만들기

각 split 폴더를 돌면서 영상 경로와 정수 라벨 리스트를 만든다.

In [ ]:
def list_split(split: str):
    paths, labels = [], []
    for cls in CLASSES:
        for p in sorted((DATA_DIR / split / cls).glob("*.avi")):
            paths.append(p)
            labels.append(class_to_idx[cls])
    return paths, labels


train_paths, train_labels = list_split("train")
val_paths, val_labels = list_split("val")
test_paths, test_labels = list_split("test")
print(f"train={len(train_paths)}, val={len(val_paths)}, test={len(test_paths)}")


## 2-2. Dataset / DataLoader

`VideoClipDataset`은 `__getitem__`에서 그때그때 영상을 열어 16프레임을 샘플링하고
Kinetics-400 통계로 정규화한 `(C=3, T=16, H=112, W=112)` 텐서를 돌려준다.

In [ ]:
import torch
from torch.utils.data import DataLoader

NUM_FRAMES = 16
CLIP_SIZE = 112
BATCH_SIZE = 4  # 6GB VRAM 기준 r3d_18 + 16프레임 클립에 맞춘 작은 배치

train_ds = VideoClipDataset(train_paths, train_labels, NUM_FRAMES, CLIP_SIZE)
val_ds = VideoClipDataset(val_paths, val_labels, NUM_FRAMES, CLIP_SIZE)
test_ds = VideoClipDataset(test_paths, test_labels, NUM_FRAMES, CLIP_SIZE)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

xb, yb = next(iter(train_loader))
print("배치 텐서:", xb.shape, "| 라벨:", yb)


## 2-3. 배치에서 뽑은 클립 시각화로 정상 동작 확인

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, BATCH_SIZE, figsize=(4 * BATCH_SIZE, 4))
for i, ax in enumerate(axes):
    frame0 = denormalize_for_display(xb[i, :, 0])  # 배치 i번째, 0번째 타임스텝
    ax.imshow(frame0)
    ax.set_title(CLASSES[yb[i]])
    ax.axis("off")
plt.tight_layout()
plt.show()


클립마다 모양이 전부 `(3, 16, 112, 112)`로 고정되어 있어서 바로 3D CNN에 넣을 수 있다.
다음 노트북에서는 이 입력 shape에 맞는 사전학습 모델들을 살펴본다.